In [2]:
# --- IMPORTS Y CONFIGURACIÓN INICIAL ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from google.colab import files
from io import BytesIO
import base64
import os

# --- SUBIDA DEL ARCHIVO ---
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

# --- SELECCIÓN DE COLUMNA DE INTERÉS ---
column_name = 'MONEY'  # Cambiar si la columna tiene otro nombre
data = df[column_name].dropna()
data.index = pd.RangeIndex(start=0, stop=len(data), step=1)  # Asegura índice limpio

# --- PRUEBA ADF ---
adf_result = adfuller(data)
adf_stat = adf_result[0]
p_value = adf_result[1]
estacionaria = p_value <= 0.05

# --- DIFERENCIACIÓN SI NO ES ESTACIONARIA ---
if not estacionaria:
    data_diff = data.diff().dropna()
else:
    data_diff = data

# --- AJUSTE ARIMA(1,1,1) ---
model = ARIMA(data, order=(1,1,1))
model_fit = model.fit()
forecast = model_fit.forecast(steps=5)

# --- GRAFICADO ---
plt.figure(figsize=(10,5))
plt.plot(data[-50:], label='Histórico')
plt.plot(range(len(data), len(data)+5), forecast, label='Pronóstico', marker='o')
plt.legend()
plt.title('Pronóstico ARIMA(1,1,1) a 5 pasos')
plt.xlabel('Índice')
plt.ylabel(column_name)
plt.grid(True)

# Guardar imagen como PNG
plot_path = "forecast_plot.png"
plt.savefig(plot_path)
plt.close()

# --- CREACIÓN DEL HTML ---
html_content = f"""
<html>
<head><title>Reporte ARIMA</title></head>
<body>
<h2>Resultado de la prueba ADF</h2>
<ul>
  <li>ADF Statistic: {adf_stat:.4f}</li>
  <li>p-value: {p_value:.4f}</li>
  <li>Interpretación: {'Estacionaria' if estacionaria else 'No estacionaria (se aplicó diferenciación)'}.</li>
</ul>
<h2>Pronóstico a 5 pasos</h2>
{forecast.to_frame(name='Pronóstico').to_html()}
<h2>Gráfico</h2>
<img src='forecast_plot.png' width='700'>
</body>
</html>
"""

# Guardar HTML
html_path = "arima_report.html"
with open(html_path, "w") as f:
    f.write(html_content)

# --- DESCARGA DE ARCHIVOS ---
files.download(html_path)
files.download(plot_path)

Saving 21.1.xlsx to 21.1 (1).xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>